# 03 · Database & SQL analysis

PostgreSQL is not installed on this machine, so the project uses **SQLite** through SQLAlchemy (`data/processed/nb15_threat.db`). `src/db.py`:

1. creates the schema from `sql/schema.sql` (fact `network_flows`, `dim_attack_type` with severity weights, `segment_summary` as the IP-summary substitute),
2. loads the cleaned data in 50k-row chunks and verifies the row count,
3. runs the 19 queries in `sql/analysis_queries.sql` (CTEs, RANK, LAG, NTILE, rolling windows) and creates the dashboard views from `sql/views.sql`,
4. exports each result to `powerbi/exports/sql/`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
SRC = Path.cwd().parent / 'src' if (Path.cwd().parent / 'src').exists() else Path.cwd() / 'src'
sys.path.insert(0, str(SRC))
import pandas as pd, json
from IPython.display import Image, Markdown, display
from config import FIG_DIR, METRICS_DIR, REPORTS_DIR, EXPORT_DIR
pd.set_option('display.max_columns', 60, 'display.width', 200)
def show(name, width=900):
    display(Image(filename=str(FIG_DIR / f'{name}.png'), width=width))

In [2]:
import db
results = db.run()


PHASE 4 - DATABASE + SQL


 - schema created


   loaded 50,000/257,673


   loaded 100,000/257,673


   loaded 150,000/257,673


...

   q16_high_rate_flows: 10 rows


   q17_land_attack_indicator: 2 rows


   q18_split_distribution: 10 rows


   q19_data_quality_flags: 5 rows


Ran 19 queries; CSVs in C:\Users\anmol\OneDrive\Desktop\project 1\powerbi\exports\sql


## Selected results

In [3]:
for q in ['q01_attack_vs_normal','q04_attack_rate_by_state','q05_attack_types_with_severity','q06_top_targeted_services','q07_suspicious_segments']:
    display(Markdown(f'### {q}'))
    display(results[q].head(12))

### q01_attack_vs_normal

,traffic,flows,pct
0,Attack,164673,63.91
1,Normal,93000,36.09


### q04_attack_rate_by_state

,state,flows,attack_flows,attack_rate_pct
0,FIN,117164,55817,47.64
1,INT,116438,106238,91.24
2,CON,20134,1402,6.96
3,REQ,3833,1201,31.33
4,RST,84,12,14.29
5,ECO,12,0,0.00
6,ACC,4,2,50.00
7,CLO,1,1,100.00
8,NO,1,0,0.00
9,PAR,1,0,0.00


### q05_attack_types_with_severity

,attack_cat,severity_level,severity_weight,flows,pct_of_attacks,weighted_impact
0,Exploits,High,0.85,44525,27.04,37846.3
1,Generic,Medium,0.60,58871,35.75,35322.6
2,DoS,High,0.75,16353,9.93,12264.8
3,Fuzzers,Low,0.40,24246,14.72,9698.4
4,Reconnaissance,Medium,0.50,13987,8.49,6993.5
5,Backdoor,Critical,0.95,2329,1.41,2212.5
6,Shellcode,Critical,0.90,1511,0.92,1359.9
7,Analysis,Low,0.45,2677,1.63,1204.7
8,Worms,Critical,1.00,174,0.11,174.0


### q06_top_targeted_services

,service,attack_flows,dominant_attack,dominant_flows,dominant_pct
0,dns,58100,Generic,57278,98.6
1,http,17650,Exploits,11481,65.0
2,smtp,4695,Exploits,4162,88.6
3,ftp,3004,Exploits,2115,70.4
4,ftp-data,1890,Exploits,1876,99.3
5,pop3,1524,Exploits,1435,94.2
6,dhcp,120,Exploits,64,53.3
7,snmp,108,Reconnaissance,80,74.1
8,ssl,86,Exploits,52,60.5
9,irc,30,Exploits,12,40.0


### q07_suspicious_segments

,proto,service,state,flows,attack_flows,attack_rate_pct,distinct_attack_types
0,udp,dns,INT,57692,57637,99.90,5
1,tcp,none,FIN,69814,26976,38.64,9
2,tcp,http,FIN,26972,17623,65.34,8
3,unas,none,INT,15599,15599,100.00,7
4,udp,none,INT,18405,12014,65.28,7
5,tcp,smtp,FIN,6844,4633,67.69,5
6,tcp,ftp,FIN,4960,2990,60.28,5
7,ospf,none,INT,2294,2289,99.78,7
8,tcp,ftp-data,FIN,5383,1888,35.07,2
9,tcp,pop3,FIN,1520,1516,99.74,3


### Window functions: LAG over duration bins and rolling 3-bin attack rate over byte deciles
Without timestamps, ordered behavioural bins replace the time axis.

In [4]:
display(results['q13_duration_bins_lag'])
display(results['q14_bytes_ntile_rolling'].head(10))

,dur_bin,flows,attack_rate_pct,prev_bin_rate,change_pts
0,0: 0s,3607,1.77,NaN,NaN
1,1: <1ms,112982,91.30,1.77,89.53
2,2: 1-10ms,18109,0.12,91.30,-91.18
3,3: 10-100ms,18471,1.17,0.12,1.05
4,4: 0.1-1s,59503,56.00,1.17,54.83
5,5: 1-10s,39420,61.00,56.00,5.00
6,6: >=10s,5581,69.06,61.00,8.06


,bytes_bin,min_sbytes,max_sbytes,flows,attack_flows,attack_rate_pct,rolling3_attack_flows,rolling3_attack_rate_pct
0,1,24,114,12884,4695,36.44,4695,36.44
1,2,114,114,12884,12884,100.00,17579,68.22
2,3,114,114,12884,12884,100.00,30463,78.81
3,4,114,114,12884,12884,100.00,38652,100.00
4,5,114,114,12884,12884,100.00,38652,100.00
5,6,114,146,12884,4933,38.29,30701,79.43
6,7,146,200,12884,9502,73.75,27319,70.68
7,8,200,200,12884,12884,100.00,27319,70.68
8,9,200,200,12884,12884,100.00,35270,91.25
9,10,200,528,12884,7482,58.07,33250,86.02


### Scan-like and DoS-style behaviour from connection-context counters
The scan-like rule is deliberately reported even though it is weak - most flows it catches are normal.

In [5]:
display(results['q08_scan_like_behaviour'])
display(results['q10_dos_patterns'])

,attack_cat,scan_like_flows,pct_of_scan_like,avg_pair_conns,avg_spkts
0,Normal,1711,55.93,21.3,4.4
1,Exploits,372,12.16,16.2,10.2
2,DoS,319,10.43,14.8,6.4
3,Generic,294,9.61,23.1,2.4
4,Fuzzers,142,4.64,13.0,4.0
5,Reconnaissance,135,4.41,18.4,4.2
6,Backdoor,42,1.37,15.4,8.6
7,Analysis,37,1.21,14.1,2.0
8,Shellcode,7,0.23,21.6,2.0


,attack_cat,flows,hot_destination_flows,hot_destination_pct,avg_pkt_rate,avg_sbytes
0,Generic,58871,38698,65.73,209833.0,446.0
1,Fuzzers,24246,846,3.49,67276.0,6764.0
2,Normal,93000,2811,3.02,19588.0,4092.0
3,Shellcode,1511,9,0.60,99146.0,533.0
4,Worms,174,1,0.57,20874.0,2256.0
5,Reconnaissance,13987,68,0.49,99719.0,751.0
6,Exploits,44525,197,0.44,67963.0,29655.0
7,DoS,16353,32,0.20,146166.0,18361.0
8,Backdoor,2329,3,0.13,149210.0,1521.0
9,Analysis,2677,0,0.00,147808.0,840.0
